# 03 — Full RAG Pipeline: Dual-Corpus Retrieval + BioMistral Generation

This notebook combines:
1. **Dual retrieval**: PubMedQA (clinical) + MedQuAD (patient-facing) via FAISS
2. **Generation**: BioMistral-7B (4-bit quantized) synthesizes grounded answers
3. **Three retrieval modes**: clinical-only, patient-only, combined

In [ ]:
# !pip install -q transformers datasets faiss-cpu sentence-transformers torch tqdm bitsandbytes accelerate peft

In [ ]:
import sys
sys.path.append('..')

import torch
import faiss
import numpy as np
from datasets import load_dataset
from src.data_utils import load_pubmedqa, filter_health_domain
from src.retriever import BiomedRetriever
from src.generator import RAGGenerator
import time

## 1. Build Dual Retrieval Indexes

In [ ]:
# ── CORPUS 1: PubMedQA (clinical evidence) ──────────────────────────
artificial_ds = load_pubmedqa('pqa_artificial')
health_examples = filter_health_domain(artificial_ds['train'])

clinical_texts = []
clinical_meta = []
for ex in health_examples:
    if isinstance(ex.get('context'), dict):
        for ctx in ex['context'].get('contexts', []):
            clinical_texts.append(ctx)
            clinical_meta.append({
                'pubid': ex.get('pubid', ''),
                'question': ex.get('question', ''),
                'source': 'PubMedQA (clinical)',
            })

print(f"Clinical corpus: {len(clinical_texts)} abstracts")

In [ ]:
# ── CORPUS 2: MedQuAD (patient-facing) ──────────────────────────────
print("Loading MedQuAD (patient-facing corpus)...")
medquad = load_dataset("lavita/MedQuAD", split="train")
print(f"MedQuAD size: {len(medquad)}")
print(f"Columns: {medquad.column_names}")
print(f"\nExample:\n  Q: {medquad[0]['question']}\n  A: {medquad[0]['answer'][:200]}...")

patient_texts = []
patient_meta = []
for item in medquad:
    if item['answer']:
        full_text = item['question'] + ' ' + item['answer']
        patient_texts.append(full_text)
        patient_meta.append({
            'question': item['question'],
            'answer': item['answer'],
            'source': 'MedQuAD (patient)',
        })

print(f"\nPatient corpus: {len(patient_texts)} documents")

In [ ]:
# Initialize the bi-encoder
retriever = BiomedRetriever(
    model_name='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
)

# Limit corpus sizes for Colab free tier memory
MAX_CLINICAL = 10000
MAX_PATIENT = 10000

# Build clinical index
print("\nBuilding clinical index...")
clinical_subset = clinical_texts[:MAX_CLINICAL]
clinical_embeddings = retriever.encode(clinical_subset)
clinical_index = faiss.IndexFlatIP(clinical_embeddings.shape[1])
clinical_index.add(clinical_embeddings)
print(f"Clinical FAISS index: {clinical_index.ntotal} vectors")

# Build patient index
print("\nBuilding patient index...")
patient_subset = patient_texts[:MAX_PATIENT]
patient_embeddings = retriever.encode(patient_subset)
patient_index = faiss.IndexFlatIP(patient_embeddings.shape[1])
patient_index.add(patient_embeddings)
print(f"Patient FAISS index: {patient_index.ntotal} vectors")

In [ ]:
def retrieve_dual(query, k=3):
    """Retrieve from both corpora."""
    q_emb = retriever.encode([query], show_progress=False)

    # Clinical retrieval
    c_scores, c_idxs = clinical_index.search(q_emb, k)
    clinical_results = []
    for score, idx in zip(c_scores[0], c_idxs[0]):
        if idx < len(clinical_subset):
            clinical_results.append({
                'text': clinical_subset[idx],
                'score': float(score),
                'meta': clinical_meta[idx],
            })

    # Patient retrieval
    p_scores, p_idxs = patient_index.search(q_emb, k)
    patient_results = []
    for score, idx in zip(p_scores[0], p_idxs[0]):
        if idx < len(patient_subset):
            patient_results.append({
                'text': patient_subset[idx],
                'score': float(score),
                'meta': patient_meta[idx],
            })

    return clinical_results, patient_results


# Quick test
query = "Why do I feel anxious and moody before my period?"
clinical, patient = retrieve_dual(query)
print(f"Query: {query}\n")
print("Clinical top result:")
print(f"  {clinical[0]['text'][:200]}...\n")
print("Patient top result:")
print(f"  {patient[0]['text'][:200]}...")

## 2. Load the Generator (BioMistral-7B, 4-bit)

In [ ]:
# Load quantized generator
generator = RAGGenerator(
    model_name='BioMistral/BioMistral-7B',
    load_in_4bit=True,
)
print("Generator loaded.")

## 3. Run the Full RAG Pipeline (Three Modes)

In [ ]:
def run_rag(question, mode='combined', k=3):
    """
    Run RAG with three retrieval modes:
    - 'clinical': PubMedQA only
    - 'patient': MedQuAD only
    - 'combined': both corpora
    """
    clinical_results, patient_results = retrieve_dual(question, k=k)

    if mode == 'clinical':
        docs = [r['text'] for r in clinical_results]
    elif mode == 'patient':
        docs = [r['text'] for r in patient_results]
    else:  # combined
        # Interleave: take top from each, ranked by score
        all_results = [
            *[{**r, 'corpus': 'clinical'} for r in clinical_results],
            *[{**r, 'corpus': 'patient'} for r in patient_results],
        ]
        all_results.sort(key=lambda x: x['score'], reverse=True)
        docs = [r['text'] for r in all_results[:k]]

    result = generator.generate_answer(question, docs)
    result['mode'] = mode
    return result

In [ ]:
# Compare all three modes on sample questions
test_questions = [
    "Does metformin improve fertility in women with PCOS?",
    "Why do I feel anxious and moody before my period?",
    "Is there a connection between endometriosis and infertility?",
    "Can birth control pills help with acne?",
    "What causes hot flashes during menopause?",
]

all_results = []
for question in test_questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print(f"{'='*80}")

    for mode in ['clinical', 'patient', 'combined']:
        result = run_rag(question, mode=mode)
        all_results.append(result)
        print(f"\n--- {mode.upper()} ---")
        print(f"Decision: {result['decision']}")
        print(f"Answer: {result['answer'][:300]}...")

## 4. Run on Labeled Evaluation Set

In [ ]:
# Load labeled evaluation data
labeled_ds = load_pubmedqa('pqa_labeled')
health_labeled = filter_health_domain(labeled_ds['train'])
print(f"Evaluation examples: {len(health_labeled)}")

# Run RAG on all labeled examples (all three modes)
eval_results = {'clinical': [], 'patient': [], 'combined': []}

for i, ex in enumerate(health_labeled):
    if i % 10 == 0:
        print(f"Processing {i}/{len(health_labeled)}...")

    question = ex['question']
    true_label = ex['final_decision']

    for mode in ['clinical', 'patient', 'combined']:
        result = run_rag(question, mode=mode, k=3)
        result['true_decision'] = true_label
        eval_results[mode].append(result)

print(f"\nDone! Processed {len(health_labeled)} examples x 3 modes.")

In [ ]:
# Save results for evaluation notebook
import json

for mode, results in eval_results.items():
    with open(f'../data/eval_results_{mode}.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Saved {len(results)} results for mode: {mode}")

## Next: Notebook 04 — Evaluation & Visualizations

Compare all three retrieval modes quantitatively and qualitatively.